In [ ]:

!pip install ultralytics pandas opencv-python-headless -q

!unzip -q /content/sample_data/data.zip -d /content/dataset/

!nvidia-smi

In [ ]:
import pandas as pd
import numpy as np
import cv2
import os
import shutil
import random


base_out = '/content/datasets/tubes'
for split in ['train', 'val']:
    os.makedirs(f'{base_out}/{split}/images', exist_ok=True)
    os.makedirs(f'{base_out}/{split}/labels', exist_ok=True)

df = pd.read_csv('/content/dataset/annotations.csv')

img_dir = '/content/dataset/images'
image_width, image_height = 640, 480

#testsplit
unique_images = df['image'].unique().tolist()
random.seed(42)
random.shuffle(unique_images)

split_idx = int(len(unique_images) * 0.8)
train_imgs = set(unique_images[:split_idx])


for image_name, group in df.groupby('image'):
    split = 'train' if image_name in train_imgs else 'val'

    label_path = f"{base_out}/{split}/labels/{image_name.replace('.png', '.txt')}"
    img_src_path = os.path.join(img_dir, image_name)
    img_dst_path = f"{base_out}/{split}/images/{image_name}"

    if os.path.exists(img_src_path):
        shutil.copy(img_src_path, img_dst_path)
    else:
        print(f"Warning: Missing image {img_src_path}")
        continue

    with open(label_path, 'w') as f:
        for _, row in group.iterrows():
            # Calculate center point
            cx = row['bbox_x'] + (row['bbox_w'] / 2)
            cy = row['bbox_y'] + (row['bbox_h'] / 2)
            w = row['bbox_w']
            h = row['bbox_h']

            angle = row['bbox_rotation']

            # Get 4 corners
            rect = ((cx, cy), (w, h), angle)
            box = cv2.boxPoints(rect)

            # Normalize coordinates to 0.0 - 1.0
            box[:, 0] /= image_width
            box[:, 1] /= image_height

            coords = box.flatten().tolist()

            # Clip to prevent YOLO crashes
            coords = [max(0.0, min(1.0, c)) for c in coords]

            coords_str = " ".join([f"{c:.6f}" for c in coords])
            f.write(f"0 {coords_str}\n")

print(f"YOLO dataset ready at: {base_out}")

In [ ]:
yaml_content = """
train: /content/datasets/tubes/train/images
val: /content/datasets/tubes/val/images

nc: 1
names: ['tube']
"""
with open('/content/data.yaml', 'w') as f:
    f.write(yaml_content)

print("data.yaml successfully created!")

In [ ]:
from ultralytics import YOLO

# Load the nano OBB model
model = YOLO('yolov8n-obb.pt')

# Train the model with heavy augmentation
results = model.train(
    data='/content/data.yaml',
    epochs=150,
    imgsz=640,
    device=0,
    batch=16,

    # Augmentations to prevent overfitting
    degrees=180.0,
    scale=0.5,
    fliplr=0.5,
    flipud=0.5,
    hsv_s=0.5,
    hsv_v=0.4,
    mosaic=1.0,

    patience=30
)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO
import glob
import random

#model loading
best_model_path = '/content/runs/obb/train/weights/best.pt'
model = YOLO(best_model_path)

#random validation images
val_images = glob.glob('/content/datasets/tubes/val/images/*.png')
sample_images = random.sample(val_images, min(3, len(val_images)))

plt.figure(figsize=(20, 7))

for i, img_path in enumerate(sample_images):
    results = model.predict(img_path, conf=0.5, verbose=False)

    #original image
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    if len(results[0].obb) > 0:

        corners = results[0].obb.xyxyxyxy.cpu().numpy()

        for box in corners:
            pts = np.int32(box)
            pts = pts.reshape((-1, 1, 2))

            cv2.polylines(img, [pts], isClosed=True, color=(0, 255, 0), thickness=2)
    # -------------------------------------------------

    plt.subplot(1, 3, i+1)
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Validation Image {i+1} Predictions")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from ultralytics import YOLO
import math
import os

#ground truth csv
df = pd.read_csv('/content/dataset/annotations.csv')
model = YOLO('/content/runs/obb/train/weights/best.pt')

val_images = list(set([f for f in os.listdir('/content/datasets/tubes/val/images/') if f.endswith('.png')]))
total_tubes_matched = 0
total_angle_error = 0.0

print("Calculating Angle Error on Validation Set...\n")

for img_name in val_images:

    gt_tubes = df[df['image'] == img_name]


    img_path = f'/content/datasets/tubes/val/images/{img_name}'
    results = model.predict(img_path, conf=0.5, verbose=False)


    if len(results[0].obb) > 0:

        preds = results[0].obb.xywhr.cpu().numpy()

        for pred in preds:
            pred_cx, pred_cy = pred[0], pred[1]


            pred_angle_deg = math.degrees(pred[4])


            best_match_dist = float('inf')
            true_angle = 0

            for _, row in gt_tubes.iterrows():
                # Ground truth center
                true_cx = row['bbox_x'] + (row['bbox_w']/2)
                true_cy = row['bbox_y'] + (row['bbox_h']/2)

                dist = math.hypot(pred_cx - true_cx, pred_cy - true_cy)
                if dist < best_match_dist:
                    best_match_dist = dist
                    true_angle = row['angle_deg']

            # If we matched a tube within 50 pixels
            if best_match_dist < 50:
                total_tubes_matched += 1

                #Calculate symmetric angle error (ignoring 180-degree flips)
                diff = abs(pred_angle_deg - true_angle) % 180
                error = min(diff, 180 - diff)

                total_angle_error += error

if total_tubes_matched > 0:
    mae = total_angle_error / total_tubes_matched
    print(f"\n Evaluated {total_tubes_matched} tubes.")
    print(f" Mean Absolute Angle Error (Symmetric): {mae:.2f} degrees")
else:
    print("\n Could not match any tubes.")

In [ ]:
# Print the raw data for downstream tasks
if len(results[0].obb) > 0:
    print("\n--- RAW TUBE DATA ---")
    predictions = results[0].obb.xywhr.cpu().numpy()

    for i, pred in enumerate(predictions):
        cx, cy, w, h, angle_rad = pred
        angle_deg = np.degrees(angle_rad)
        print(f"Tube {i+1}: Center(X:{cx:.1f}, Y:{cy:.1f}) | Angle: {angle_deg:.1f}°")

In [ ]:
import os
import cv2
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Configuration & Setup
model_path = '/content/runs/obb/train/weights/best.pt'
input_dir = '/content/datasets/tubes/val/images/'
output_img_dir = '/content/annotated_validation_images/'
output_csv_path = '/content/final_predictions.csv'

# Create the output directory for the images if it doesn't exist
os.makedirs(output_img_dir, exist_ok=True)

# Load the trained YOLO model
model = YOLO(model_path)
output_data = []

print(f"Processing images in {input_dir}...")

# 2. Iterate through all images
image_files = [f for f in os.listdir(input_dir) if f.endswith('.png')]

for img_name in image_files:
    img_path = os.path.join(input_dir, img_name)

    # Run Inference
    results = model.predict(img_path, conf=0.5, verbose=False)

    # Load image for OpenCV drawing
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    if len(results[0].obb) > 0:
        # Get raw data for CSV math
        preds_xywhr = results[0].obb.xywhr.cpu().numpy()

        # Get the exact 4 corners for drawing the boxes
        corners = results[0].obb.xyxyxyxy.cpu().numpy()

        # --- A. APPEND TO CSV DATA ---
        for pred in preds_xywhr:
            cx, cy, w, h, angle_rad = pred

            bbox_rotation = math.degrees(angle_rad)
            bbox_x = cx - (w / 2)
            bbox_y = cy - (h / 2)
            angle_deg = bbox_rotation % 180

            output_data.append({
                'image': img_name,
                'center_x': round(cx, 2),
                'center_y': round(cy, 2),
                'bbox_x': round(bbox_x, 2),
                'bbox_y': round(bbox_y, 2),
                'bbox_w': round(w, 2),
                'bbox_h': round(h, 2),
                'bbox_rotation': round(bbox_rotation, 2),
                'angle_deg': round(angle_deg, 2)
            })

        # --- B. DRAW ON IMAGE (MINIMALIST BOXES ONLY) ---
        for box in corners:
            # Convert float coordinates to integers for drawing
            pts = np.int32(box).reshape((-1, 1, 2))

            # Draw a clean, bright green bounding box outlining the tube
            cv2.polylines(img, [pts], isClosed=True, color=(0, 255, 0), thickness=2)

    # Save the minimalist image
    save_path = os.path.join(output_img_dir, img_name)
    cv2.imwrite(save_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

# 3. Save the master CSV
df_output = pd.DataFrame(output_data)
if not df_output.empty:
    columns_order = ['image', 'center_x', 'center_y', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'bbox_rotation', 'angle_deg']
    df_output = df_output[columns_order]
    df_output.to_csv(output_csv_path, index=False)

print(f"\n✅ Pipeline Complete!")
print(f"📁 Extracted {len(df_output)} tubes into {output_csv_path}")
print(f"🖼️ Saved {len(image_files)} minimalist annotated images to {output_img_dir}")

# 4. Display one example image in the notebook just to verify
if len(image_files) > 0:
    example_img = cv2.imread(os.path.join(output_img_dir, image_files[0]))
    plt.figure(figsize=(12, 10))
    plt.imshow(cv2.cvtColor(example_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title("Example Output (Boxes Only)")
    plt.show()